## Agents and tools using LCEL

In [8]:
import langchain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

google_llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model="gemini-2.0-flash", 
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0, 
    model="gpt-4", 
    api_key=openai_api_key
)


In [9]:
from langchain.agents import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

pdf_loader_1 = PyPDFLoader(
    "./docs_for_rag/for_agents_lectures/wellarchitected-framework.pdf",
)

pdf_loader_2 = PyPDFLoader(
    "./docs_for_rag/for_agents_lectures/gzip.pdf",
)

text_loader = TextLoader(
    "./docs_for_rag/for_agents_lectures/coolie_english.txt",
)

pdf_1_docs = pdf_loader_1.load()
pdf_2_docs = pdf_loader_2.load()
text_docs = text_loader.load()

all_docs = pdf_1_docs + pdf_2_docs + text_docs

In [10]:
print(len(pdf_1_docs))
print(len(pdf_2_docs))
print(len(text_docs))

# all_docs = text_docs

999
29
1


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(all_docs)

In [12]:
len(split_docs)

1476

In [13]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(split_docs, embeddings)

In [ ]:
results = await vectorstore.asimilarity_search("who is dahaa in coolie?")
                                        

4

In [ ]:
retriever = vectorstore.as_retriever()

results = retriever.invoke("who is dahaa in coolie?")

4

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", """Always answer the question just by using the context provided and not from your knowledge.
        Context: {context}
        question: {input}
     
        Answer: 
     """),
     ("placeholder", "{agent_scratchpad}")
])


chain = {"context": retriever, "input": RunnablePassthrough()} | prompt | google_llm

# chain = {"context": RunnableLambda(lambda x: x["input"]) | retriever, "input": RunnableLambda(lambda x: x["input"])} | prompt | google_llm | StrOutputParser()

res = chain.invoke("who is dahaa in coolie movie?")

res

AIMessage(content='Daaha is a mysterious figure who leads a larger global network.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--8eff507a-a08c-46cd-bd25-1c98a248fcc6-0', usage_metadata={'input_tokens': 3174, 'output_tokens': 15, 'total_tokens': 3189, 'input_token_details': {'cache_read': 0}})